# NOVAMART — Phase 8 ABC and Supplier Analysis

ABC uses 2023 source synthetic sales × listed price. Supplier outcomes and scores come from a separately **SIMULATED NOVAMART PROCUREMENT SCENARIO**. The weights and tiers are assumed. No Power BI dashboard is built.

## 1. Inputs, complete-year value basis, and ABC rule
For each of 100 local SKUs, sum daily `units_sold × listed_price` over January–December 2023. This is a sales-value proxy, not audited revenue or procurement cost. Sort descending and assign the crossing SKU to the preceding A (80%) or B (95%) band.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd() if (Path.cwd()/"data/processed/daily_operations.csv").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/"src"))
from abc_supplier_analysis import build
summary=build()
abc=pd.read_csv(ROOT/"reports/abc_inventory_classification.csv")
print(summary)
display(abc.head())
display(Image(filename=str(ROOT/"images/abc_supplier/01_abc_cumulative.png")))
display(Image(filename=str(ROOT/"images/abc_supplier/02_abc_count_value.png")))

## 2. ABC and replenishment risk
ABC is economic value importance. It does not directly measure demand uncertainty or determine safety stock. Join once to the Phase 7 one-row-per-local-SKU recommendation; preserve the original order priority separately.

In [ ]:
matrix=pd.read_csv(ROOT/"reports/supply_chain_priority_matrix.csv")
display(pd.crosstab(matrix.abc_class,matrix.inventory_status))
display(Image(filename=str(ROOT/"images/abc_supplier/03_reorder_by_abc.png")))

## 3. SIMULATED supplier KPIs and score
On-time, fill, actual lead-time population SD, and positive delay are calculated from joined simulated POs and receipts. Min-max scores favor higher on-time/fill and lower variability/delay. Base weights are 40/30/20/10, sum to 100%; spend is exposure, excluded from score.

In [ ]:
sup=pd.read_csv(ROOT/"reports/supplier_scorecard.csv")
display(sup[["supplier_id","purchase_orders","on_time_pct","receipt_fill_pct","avg_actual_lead_days","lead_sd_days","supplier_score_assumed_weights","supplier_tier"]].round(2))
display(Image(filename=str(ROOT/"images/abc_supplier/04_supplier_on_time_fill.png")))
display(Image(filename=str(ROOT/"images/abc_supplier/05_supplier_scores.png")))

## 4. Weight sensitivity and management priority
Alternative weights 25/35/25/15 are tested. Priority combines value class, Phase 7 condition, and a supplier tier; it is a review queue, not a measured financial loss or an automatic sourcing decision.

In [ ]:
display(sup[["supplier_id","base_rank","alternative_rank","supplier_score_assumed_weights","supplier_score_alternative_weights"]].round(2))
display(matrix.management_priority.value_counts().to_frame("local_skus"))
display(matrix[matrix.management_priority.eq("CRITICAL")][["local_sku_id","abc_class","supplier_id","inventory_status","sales_value_proxy_2023","recommended_order_quantity"]].head())
display(Image(filename=str(ROOT/"images/abc_supplier/06_priority_matrix.png")))

## 5. Validation and limits
Source is synthetic; supplier events are further simulated. The 0–100 supplier normalization is relative to only six suppliers. The 97 Phase 7 reorder flags cause few routine cases. The Power BI relationship guide prepares Phase 9 without building a dashboard.

In [ ]:
assert len(abc)==abc.local_sku_id.nunique()==100
assert len(sup)==6 and len(matrix)==matrix.local_sku_id.nunique()==100
assert abs(abc.value_share_pct.sum()-100)<.001
print("Notebook assertions passed.")